In [ ]:
!pip install -q langchain langchain-openai

In [ ]:
from google.colab import userdata
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain.tools import ToolRuntime, tool
from langchain_core.messages import BaseMessage
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from pydantic import SecretStr
from typing import List, TypedDict

openai_api_key = SecretStr(userdata.get('OPENAI_API_KEY'))

def print_conversation(conversation: List[BaseMessage]):
    for message in conversation:
        message.pretty_print()

In [ ]:
class AgentContext(TypedDict):
    user_id: str

In [ ]:
checkpointer = InMemorySaver()
store = InMemoryStore()

In [ ]:
@tool
def remember_user_facts(key: str, value: str, runtime: ToolRuntime[AgentContext]) -> str:
    """
    Extract durable user facts from a user message and store them in long-term memory. Example: "key: allergy; value: The user is allergic to nuts.", "key: hobbies; value: The user can play the guitar."

    Args:
        key: A unique identifier of the fact.
        value: The fact itself.
    """
    namespace = ("users", runtime.context["user_id"], "facts")
    runtime.store.put(namespace, key, { "value": value })
    return "OK"

@tool
def recall_user_facts(runtime: ToolRuntime[AgentContext]) -> str:
    """
    Recall previously stored long-term facts about the user.
    """
    namespace = ("users", runtime.context["user_id"], "facts")
    facts = runtime.store.search(namespace, limit=20)
    if not facts:
        return "No facts stored."

    return f"Known facts: {'; '.join(f'{fact.key}: \"{fact.value["value"]}\"' for fact in sorted(facts, key=lambda x: x.key))}"

In [ ]:
agent = create_agent(
    model=ChatOpenAI(model="gpt-5-nano", api_key=openai_api_key, reasoning_effort="low"),
    tools=[remember_user_facts, recall_user_facts],
    system_prompt="""
                  You are a polite and helpful personal assistant.
                  You should use frequently the \"remember_user_facts\" tool to store information about the user that can be useful in future.
                  At the start of each iteration, ALWAYS use the \"recall_user_facts\" tool.
                  Be friendly - include known facts in the conversation to make the user feel special.
                  """,
    checkpointer=checkpointer,
    store=store
)

In [ ]:
introduction_response = agent.invoke(
    input={
        "messages": [HumanMessage("Hello! My name is Tony and I would like you to help me with the management of my personal notes and timeline.")]
    },
    config={
        "configurable": {
            "thread_id": "1"
        }
    },
    context={
        "user_id": "troeff"
    }
)

In [ ]:
print_conversation(introduction_response["messages"])

In [ ]:
# NOTE: We are passing a different `thread_id` - this is a different conversation (a few days after)
# The expected result - the agent should know (at least) my name.
plan_meeting_response = agent.invoke(
    input={
        "messages": [HumanMessage("I want to plan a business meeting for today, 15:00.")]
    },
    config={
        "configurable": {
            "thread_id": "2"
        }
    },
    context={
        "user_id": "troeff"
    }
)

In [ ]:
print_conversation(plan_meeting_response["messages"])